# Historical experiment — do not use its scores as validated results
This notebook applies SMOTE before splitting and depends on missing Drive data. Use the maintained `train.py` baseline. Dataset provenance and original CNN reproduction remain unverified.


<a href="https://colab.research.google.com/github/meahmadkarim/Expression_Classification_/blob/main/Expression_Classification_from_Facial_Images_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/AI Project/data/image

In [ ]:
!7za -y x "/content/drive/MyDrive/AI Project/data/image/origin.7z*"

In [ ]:
!cp -r ./origin "/content/drive/MyDrive/AI Project"

In [ ]:
label_file=r"/content/drive/MyDrive/AI Project/data/label/label.lst"
images_file=r"/content/drive/MyDrive/AI Project/origin"

In [ ]:
from numpy import split
import pandas as pd
data= pd.read_csv(label_file, sep=" ",header=None)
columns="image_name face_id_in_image face_box_top face_box_left face_box_right face_box_bottom face_box_confidence expression_label".split()
data.columns = columns

data.head(10)

In [ ]:
data.shape

In [ ]:
data_cof=data[data.face_box_confidence>50]

In [ ]:
data_cof.shape

In [ ]:
import cv2
import os
total_img=[]
total_lab=[]
for i, row in data_cof.sample(10000).iterrows():
  image_name=row["image_name"]
  x = row["face_box_left"]
  y=row["face_box_top"]
  w=row["face_box_right"]
  z=row["face_box_bottom"]
  label=row["expression_label"]
  img_path=os.path.join(images_file,image_name)

  image = cv2.imread(img_path)

  #croping
  if image is not None:
        crop = image [y:z, x:w]
        img_resize = cv2.resize(crop, (64, 64))
        if img_resize is not None:
         total_img.append(img_resize)
         total_lab.append(label)


In [ ]:
import numpy as np
X = np.array(total_img)
Y = np.array(total_lab)

In [ ]:
len(Y)



In [ ]:
len(X)

In [ ]:
x = X / 255.0

In [ ]:
X_reshape = x.reshape((-1,64*64*3))
X_reshape.shape

In [ ]:
import pandas as pd
pd.Series(Y).value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE
sampler = SMOTE()
X_smote,Y_smote=sampler.fit_resample(X_reshape, Y)

In [ ]:
x = X_smote.reshape(-1,64,64,3)

In [ ]:
len(Y_smote)

In [ ]:
z = Y_smote

In [ ]:
import numpy as np
from keras.utils import to_categorical
y = to_categorical(z)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split


X_train, X_temp, Y_train, Y_temp = train_test_split(x, y, test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)


In [ ]:
import numpy as np
from keras.preprocessing.image import ImageDataGenerator

image_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)


image_datagen.fit(X_train)


In [ ]:
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(64,64,3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
num_classes = 7
model.add(Dense(num_classes, activation='softmax'))


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(image_datagen.flow(X_train, Y_train, batch_size=32), epochs=50, validation_data=(X_val, Y_val))

In [ ]:
loss ,accuracy =model.evaluate(X_test,Y_test )
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")